In [1]:
%load_ext autoreload
%autoreload 2

import mujoco
from mujoco import MjsBody
import numpy as np
import PIL.Image

from swarmbots.unit import Unit
from rendering import display_video, display_image


In [2]:
RENDER_WIDTH = 640
RENDER_HEIGHT = 480


In [8]:

spec = mujoco.MjSpec()
worldbody: MjsBody = spec.worldbody

unit = Unit(
    body_radius=0.1,
    leg_length=0.2,
    leg_radius=0.025,
    hinge_range=np.pi / 4
)
unit.body.add_joint(type=mujoco.mjtJoint.mjJNT_FREE)

worldbody.add_frame(pos=[0, 0, 2], euler=[1, 2, 3]).attach_body(unit.body, 'Unit1--', '')


worldbody.add_geom(
    type=mujoco.mjtGeom.mjGEOM_PLANE,
    size=[2, 2, 0.1],
    rgba=[0.2, 0.3, 0.4, 1],
    pos=[0, 0, 0]
)

worldbody.add_light(pos=[0, 0, 3], dir=[0, 0, -1])
worldbody.add_light(pos=[2, 2, 3], dir=[-1, -1, -1])

# worldbody.add_camera(pos=[0, -5, 0])

model = spec.compile()
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=RENDER_HEIGHT, width=RENDER_WIDTH)

print(f"Model compiled. nq={model.nq}, nv={model.nv}")


Model compiled. nq=19, nv=18


In [9]:
print(spec.to_xml())

<mujoco model="MuJoCo Model">
  <compiler angle="radian"/>

  <default>
    <default class="Unit1--main"/>
  </default>

  <worldbody>
    <geom size="2 2 0.1" type="plane" rgba="0.2 0.3 0.4 1"/>
    <light pos="0 0 3" dir="0 0 -1"/>
    <light pos="2 2 3" dir="-0.57735 -0.57735 -0.57735"/>
    <body name="Unit1--main_body" pos="0 0 2" quat="0.999463 0.00917905 0.0172174 0.0263242">
      <joint type="free"/>
      <geom size="0.1" rgba="0.75 0 0 0.1"/>
      <body name="Unit1--limb_root_xp" pos="0.1 0 0" quat="0.707107 0 0.707107 0">
        <body name="Unit1--limb_xp">
          <joint name="Unit1--limb_xp-hinge1z" pos="0 0 0" axis="0 0 1"/>
          <geom size="0.025 0.01" pos="0 0 0.01" quat="0 1 0 0" type="cylinder" rgba="0 0 0 1"/>
          <body name="Unit1--limb_xp-seg2" pos="0 0 0.02">
            <joint name="Unit1--limb_xp-hinge2x" pos="0 0 0" axis="1 0 0" range="-0.785398 0.785398"/>
            <geom size="0.025 0.09" pos="0 0 0.09" quat="0 1 0 0" type="cylinder" rgba="0

In [5]:

# mujoco.mj_resetData(model, data)

# while data.time < 1:
#     mujoco.mj_step(model, data)

# renderer.update_scene(data, camera=-1)
# display_image(renderer.render().copy())

In [6]:
# Simple simulation loop for smoke testing
DURATION = 5.0  # seconds
FRAMERATE = 30  # Hz
frames = []

mujoco.mj_resetData(model, data)
data.qpos[0:3] = [0, 0, 1]

data.ctrl[:] = 1

while data.time < DURATION:
    mujoco.mj_step(model, data)
    if len(frames) < data.time * FRAMERATE:
        renderer.update_scene(data)
        frames.append(renderer.render().copy())

print(f"Simulated {len(frames)} frames")
display_video(frames, FRAMERATE)


Simulated 151 frames


In [7]:
mujoco.MjSpec().compiler